# Data Source: Neighborhood Context

In [1]:
import pandas as pd
import time
import osmnx as ox

from stayrank.config import PROCESSED_DIR, BOROUGHS_LIST

## Define POI tags and helpers

In [2]:
POIS_TAGS = {
    "amenity": ["restaurant", "cafe", "bar", "pub", "nightclub"],
    "leisure": ["park"]
}

def classify_poi(row):
    if row.get("amenity") in {"restaurant", "cafe"}:
        return "dining"
    if row.get("amenity") in {"bar", "pub", "nightclub"}:
        return "nightlife"
    if row.get("leisure") in {"park"}:
        return "parks"
    return "other"

def get_borough_pois_counts(borough: str) -> pd.Series:
    try:
        query = f"{borough}, Montreal, Quebec, Canada"
        gdf = ox.features_from_place(query, tags=POIS_TAGS)
        gdf = gdf.reset_index()

        if gdf.empty:
            return pd.DataFrame()
        
        gdf["poi_category"] = gdf.apply(classify_poi, axis=1)

        counts = gdf["poi_category"].value_counts()

        result = pd.Series({
            "borough": borough,
            **counts.to_dict()
        })

        return result
    except Exception as e:
        print(f"Failed for {borough}: {e}")

## Test one borough

In [3]:
get_borough_pois_counts(borough="Ville-Marie")

borough      Ville-Marie
dining               800
parks                150
nightlife            146
dtype: object

## Build neighborhood context

In [4]:
# Collect neighborhood context
records = []

for borough in BOROUGHS_LIST:
    context = get_borough_pois_counts(borough=borough)
    
    if context is None:
        print(f"Failed to process '{borough}'")
    
    records.append(context)
    time.sleep(1.5)

neighborhood_ctx = pd.DataFrame(records)

# Clean and format data
context_cols = ["dining", "parks", "nightlife"]

neighborhood_ctx[context_cols] = (
    neighborhood_ctx[context_cols]
    .fillna(0)
    .astype(int)
)

neighborhood_ctx.head()

,borough,dining,parks,nightlife
0,Ville-Marie,800,150,146
1,Rosemont-La Petite-Patrie,349,68,53
2,Côte-des-Neiges-Notre-Dame-de-Grâce,264,59,14
3,Le Sud-Ouest,189,109,35
4,Le Plateau-Mont-Royal,533,62,90


In [5]:
neighborhood_ctx.isna().sum()

borough      0
dining       0
parks        0
nightlife    0
dtype: int64

## Save processed data

In [6]:
CLEAN_NEIGHBORHOOD_CONTEXT_PATH = PROCESSED_DIR / "neighborhood_context.parquet"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

neighborhood_ctx.to_parquet(CLEAN_NEIGHBORHOOD_CONTEXT_PATH, index=False)

CLEAN_NEIGHBORHOOD_CONTEXT_PATH

WindowsPath('C:/Users/ngoum/Documents/coding/data/agentic-pyrank/data/processed/neighborhood_context.parquet')